<div style="padding: 20px; background: linear-gradient(90deg, #ff416c 0%, #ff4b2b 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">⚠️ Module 5.2: The Score Threshold Problem</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Preventing AI hallucinations by filtering out bad retrieval results.</p>
</div>

---

## 1. The Danger of `k`

If you hardcode `k=5`, your vector database will **always return 5 documents**. 
But what if the user asks a question about "Quantum Physics" and your database only contains recipes for cake? 

The database will still return the 5 *least irrelevant* cake recipes! If you feed those to an LLM, it will hallucinate. We need to enforce a **minimum similarity threshold**.

### Course alignment and free-first stack

- Covers: Similarity score thresholds and out-of-domain query rejection.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))

# A database entirely about European Landmarks
docs = [
    Document(page_content="The Eiffel Tower is located in Paris, France."),
    Document(page_content="The Colosseum is an ancient amphitheater in Rome, Italy."),
]
vs = Chroma.from_documents(docs, embeddings, collection_name="threshold_demo")
print("Database initialized.")

## 2. Implementing `similarity_score_threshold`
Using LangChain's `as_retriever()`, we can specify that we want up to `k` documents, but **only if** their relevance score exceeds a certain threshold.

In [ ]:
# The user asks a completely unrelated question
bad_query = "How do I bake a chocolate cake?"

# 1. Standard Search (Fails silently by returning irrelevant data)
std_results = vs.similarity_search(bad_query, k=2)
print("--- Standard Search Results (DANGEROUS) ---")
for d in std_results:
    print(f"Returned: {d.page_content}")

# 2. Threshold Retriever (Protects the LLM)
# Note: HuggingFace returns distance (lower is better), so in LangChain's generic 
# threshold retriever, it converts distance to a relevance score (0.0 to 1.0) internally.
threshold_retriever = vs.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'score_threshold': 0.5, 'k': 2}
)

safe_results = threshold_retriever.invoke(bad_query)
print("\n--- Threshold Search Results (SAFE) ---")
if not safe_results:
    print("SUCCESS: No documents matched the minimum threshold. The LLM can safely reply 'I don't know'.")
else:
    for d in safe_results:
        print(f"Returned: {d.page_content}")